In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 00 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 00.C1 — Read the dimensions

Return (number of observations, number of features) for a nonempty rectangular list. Do not count labels as features.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Count the outside list first, then one row. Index 0 means the first row.

</details>

In [3]:
def table_shape(rows):
    return len(rows), len(rows[0])

In [4]:
assert table_shape([[1,2],[3,4],[5,6]]) == (3,2)
X=load_iris_data().iloc[:,1:5].to_numpy()
assert table_shape(X.tolist()) == (150,4)
print("Measured Iris feature table:",table_shape(X.tolist()))

Measured Iris feature table: (150, 4)


**Why this works and what it does not establish:** A table has two different dimensions. The 150 rows are samples; the four columns are measurements of each sample. A two-dimensional array can therefore represent points in four-dimensional feature space.

## 00.C2 — Calculate a mean without a library

For a nonempty rectangular numeric table, calculate the arithmetic mean of column j using a loop and division.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Keep a running total; divide once after the loop.

</details>

In [5]:
def column_mean(rows, j):
    total=0.0
    for row in rows:
        total += row[j]
    return total/len(rows)

In [6]:
assert column_mean([[1,9],[3,7]],0)==2
X=load_iris_data().iloc[:,1:5].to_numpy()
assert np.isclose(column_mean(X,0),np.mean(X[:,0]))
print("Mean sepal length in this snapshot (cm):",column_mean(X,0))

Mean sepal length in this snapshot (cm): 5.843333333333335


**Why this works and what it does not establish:** The loop visits observations, not features. The result summarizes this supplied dataset. It does not establish the mean for every flower in the population.

## 00.C3 — Turn ink into an entrance value

The supplied digits have intensity 0..16. Return f=1-image/16 as a floating-point array so dark ink enters a sublevel filtration first.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Use / for division. Do not use //, which discards fractional parts.

</details>

In [7]:
def ink_entrance(image):
    return 1.0-np.asarray(image,dtype=float)/16.0

In [8]:
assert np.allclose(ink_entrance([[0,8,16]]),[[1,.5,0]])
image,sample_id=digit_example(0)
f=ink_entrance(image)
assert f.shape==(8,8) and f.min()>=0 and f.max()<=1
print("Training image",sample_id,"entrance range:",float(f.min()),float(f.max()))

Training image 0 entrance range: 0.0625 1.0


**Why this works and what it does not establish:** This uses the documented physical encoding range, not a mean or maximum fitted using test observations. The word sublevel means that cells with smaller values enter earlier.

## 00.C4 — Separate measurements from labels

Return X and y from the bundled Iris DataFrame. X contains exactly the four *_cm measurements; y is the label. Do not include sample_id or species in X.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Select columns by their names before converting to NumPy.

</details>

In [9]:
def iris_xy(frame):
    names=['sepal_length_cm','sepal_width_cm','petal_length_cm','petal_width_cm']
    return frame[names].to_numpy(dtype=float),frame['label'].to_numpy()

In [10]:
X,y=iris_xy(load_iris_data())
assert X.shape==(150,4) and y.shape==(150,)
assert np.array_equal(np.unique(y),[0,1,2])
print("X shape",X.shape,"y shape",y.shape)

X shape (150, 4) y shape (150,)


**Why this works and what it does not establish:** An identifier is useful for provenance, not automatically a coordinate. A label is a prediction target or an annotation. Inserting it into a supposedly unsupervised distance calculation changes the question.

## 00.C5 — Count a recorded category

Count how many entries of a one-dimensional label array equal a requested value. Use the training split only in the real-data check.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Each equality comparison contributes one for true and zero for false.

</details>

In [11]:
def count_label(labels, wanted):
    return sum(int(label==wanted) for label in labels)

In [12]:
assert count_label([0,1,0,2],0)==2
images,labels,ids=load_digit_data('train')
counts=[count_label(labels,k) for k in range(10)]
assert sum(counts)==1077
print("Training class counts:",counts)

Training class counts: [106, 109, 107, 109, 109, 109, 109, 107, 104, 108]


**Why this works and what it does not establish:** Class counts help you understand the sample distribution before a model is fitted. The check does not open the held-out images.